# Yas Customer Support RAG Chatbot

This notebook builds a customer support RAG chatbot for Yas by extracting useful information from the Yas website, collecting unique page links, scraping page titles, headings, and content, cleaning and preparing the text, converting it into documents, creating embeddings, storing them in a vector database and using the retrieved information to answer customer questions accurately with source-based responses.

In [8]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

base_url = "https://www.yas.co.tz/"

response = requests.get(base_url)
soup = BeautifulSoup(response.text, "html.parser")

links = []

for a_tag in soup.find_all("a"):
    href = a_tag.get("href")

    if href:
        full_link = urljoin(base_url, href)

        if full_link.startswith("https://www.yas.co.tz/"):
            links.append(full_link)

links = list(dict.fromkeys(links))

print("Unique links:", len(links))
print("-" * 50)

for link in links:
    try:
        page_response = requests.get(link)
        page_soup = BeautifulSoup(page_response.text, "html.parser")

        title = (
            page_soup.title.string.strip()
            if page_soup.title and page_soup.title.string
            else "No title found"
        )

        h1_tag = page_soup.find("h1")
        h1 = h1_tag.get_text(strip=True) if h1_tag else "No heading found"

        print("URL:", link)
        print("Page title:", title)
        print("Main heading:", h1)
        print("-" * 50)

    except Exception as e:
        print("URL:", link)
        print("Error:", e)
        print("-" * 50)

Unique links: 72
--------------------------------------------------
URL: https://www.yas.co.tz/#content
Page title: Consumer - Yas Tanzania
Main heading: ExperienceMtandao wa Viwango
--------------------------------------------------
URL: https://www.yas.co.tz/
Page title: Consumer - Yas Tanzania
Main heading: ExperienceMtandao wa Viwango
--------------------------------------------------
URL: https://www.yas.co.tz/business/
Page title: Business - Yas Tanzania
Main heading: Welcome toYas Business
--------------------------------------------------
URL: https://www.yas.co.tz/fiber-home/
Page title: Fiber Home - Yas Tanzania
Main heading: Where fast meets reliable,fiber connection
--------------------------------------------------
URL: https://www.yas.co.tz/about/
Page title: About - Yas Tanzania
Main heading: Leading Tanzania'sdigital future
--------------------------------------------------
URL: https://www.yas.co.tz/assistance
Page title: Assistance - Yas Tanzania
Main heading: We're h

In [10]:
with open("extracted_links.txt", "w", encoding="utf-8") as file:
    for link in links:
        file.write(link + "\n")

In [14]:
from urllib.parse import urlparse
import requests
from langchain_core.documents import Document

In [11]:
file_path = 'extracted_links.txt'  


lists=[]
with open(file_path, 'r') as file:
    for line in file:
        url = line.strip()  
        lists.append(url)

In [12]:
lists

['https://www.yas.co.tz/#content',
 'https://www.yas.co.tz/',
 'https://www.yas.co.tz/business/',
 'https://www.yas.co.tz/fiber-home/',
 'https://www.yas.co.tz/about/',
 'https://www.yas.co.tz/assistance',
 'https://www.yas.co.tz/devices/',
 'https://www.yas.co.tz/mixx-by-yas/',
 'https://www.yas.co.tz/consumer/mobile-plans/wakishua/',
 'https://www.yas.co.tz/consumer/mobile-plans/internet/',
 'https://www.yas.co.tz/consumer/mobile-plans/mix/',
 'https://www.yas.co.tz/consumer/mobile-plans/voice-sms/',
 'https://www.yas.co.tz/consumer/mobile-plans/roaming/',
 'https://www.yas.co.tz/consumer/mobile-plans/international/',
 'https://www.yas.co.tz/consumer/mobile-plans/best-deals/',
 'https://www.yas.co.tz/consumer/home-plans/',
 'https://www.yas.co.tz/consumer/lifestyle/',
 'https://www.yas.co.tz/consumer/network-of-viwango/',
 'https://www.yas.co.tz/business/pro-sme/mobile/',
 'https://www.yas.co.tz/business/pro-sme/internet-connectivity/',
 'https://www.yas.co.tz/business/pro-sme/value-

In [23]:
'''
from bs4 import BeautifulSoup
import re
def parse_html(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    return soup.get_text()

def fetch_webpage(url):
    response = requests.get(url)
    return response.text

def clean_text(text):
    # Remove extra whitespace and newlines
    text = ' '.join(text.split())
    # Remove any remaining unwanted characters or patterns (example: URLs, special characters)
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'"]+', '', text)  # Remove special characters except punctuation
    return text

from docx import Document

def write_to_docx(text_content, output_filename):
    doc = Document()  # Create a new Document object

    # Add a heading
    doc.add_heading('Document Title', level=1)

    doc.add_paragraph(text_content)

    # Save the document
    doc.save(output_filename)
'''    

'\nfrom bs4 import BeautifulSoup\nimport re\ndef parse_html(html_content):\n    soup = BeautifulSoup(html_content, \'html.parser\')\n    return soup.get_text()\n\ndef fetch_webpage(url):\n    response = requests.get(url)\n    return response.text\n\ndef clean_text(text):\n    # Remove extra whitespace and newlines\n    text = \' \'.join(text.split())\n    # Remove any remaining unwanted characters or patterns (example: URLs, special characters)\n    text = re.sub(r\'http\\S+\', \'\', text)  # Remove URLs\n    text = re.sub(r\'[^a-zA-Z0-9\\s.,!?\'"]+\', \'\', text)  # Remove special characters except punctuation\n    return text\n\nfrom docx import Document\n\ndef write_to_docx(text_content, output_filename):\n    doc = Document()  # Create a new Document object\n\n    # Add a heading\n    doc.add_heading(\'Document Title\', level=1)\n\n    doc.add_paragraph(text_content)\n\n    # Save the document\n    doc.save(output_filename)\n'

In [22]:
import requests
import time
import re
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from docx import Document
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------- HEADERS ----------------
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
}

# ---------------- SESSION WITH RETRY ----------------
session = requests.Session()

retry = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("http://", adapter)
session.mount("https://", adapter)

# ---------------- FETCH WEBPAGE (SAFE) ----------------
def fetch_webpage(url):
    try:
        response = session.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        return response.text

    except Exception as e:
        print("Failed:", url, e)
        return None

# ---------------- PARSE HTML ----------------
def parse_html(html_content):
    soup = BeautifulSoup(html_content, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.extract()

    return soup.get_text(separator=" ", strip=True)

# ---------------- CLEAN TEXT ----------------
def clean_text(text):
    text = " ".join(text.split())
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s.,!?'\"]+", "", text)
    return text

# ---------------- SAFE FILENAME ----------------
def safe_filename(url, index):
    parsed = urlparse(url)

    name = parsed.path.strip("/")

    if not name:
        name = "homepage"

    name = name.replace("/", "_")
    name = re.sub(r'[<>:"/\\|?*#%]', "_", name)

    return f"{index}_{name}.docx"

# ---------------- WRITE DOCX ----------------
def write_to_docx(text, filename):
    doc = Document()
    doc.add_heading("YAS DATA", level=1)
    doc.add_paragraph(text)
    doc.save(filename)

# ---------------- MAIN SCRAPER ----------------
success = 0
failed = 0

for i, url in enumerate(lists, start=1):

    # skip bad URLs
    if "#" in url or "cdn-cgi" in url:
        continue

    print(f"\nScraping {i}: {url}")

    html_content = fetch_webpage(url)

    if html_content is None:
        failed += 1
        continue

    page_text = parse_html(html_content)
    final_text = clean_text(page_text)

    output_filename = safe_filename(url, i)

    write_to_docx(final_text, output_filename)

    print(f"Saved: {output_filename}")

    success += 1

    time.sleep(3)  # IMPORTANT: prevents blocking

# ---------------- SUMMARY ----------------
print("\n========== DONE ==========")
print("Success:", success)
print("Failed:", failed)
print("Total files:", success + failed)


Scraping 2: https://www.yas.co.tz/
Saved: 2_homepage.docx

Scraping 3: https://www.yas.co.tz/business/
Saved: 3_business.docx

Scraping 4: https://www.yas.co.tz/fiber-home/
Saved: 4_fiber-home.docx

Scraping 5: https://www.yas.co.tz/about/
Saved: 5_about.docx

Scraping 6: https://www.yas.co.tz/assistance
Saved: 6_assistance.docx

Scraping 7: https://www.yas.co.tz/devices/
Saved: 7_devices.docx

Scraping 8: https://www.yas.co.tz/mixx-by-yas/
Saved: 8_mixx-by-yas.docx

Scraping 9: https://www.yas.co.tz/consumer/mobile-plans/wakishua/
Saved: 9_consumer_mobile-plans_wakishua.docx

Scraping 10: https://www.yas.co.tz/consumer/mobile-plans/internet/
Saved: 10_consumer_mobile-plans_internet.docx

Scraping 11: https://www.yas.co.tz/consumer/mobile-plans/mix/
Saved: 11_consumer_mobile-plans_mix.docx

Scraping 12: https://www.yas.co.tz/consumer/mobile-plans/voice-sms/
Saved: 12_consumer_mobile-plans_voice-sms.docx

Scraping 13: https://www.yas.co.tz/consumer/mobile-plans/roaming/
Saved: 13_consu

In [25]:
#convert the files from word to pdf
from docx2pdf import convert
input_folder = r"C:\Users\Admin\Desktop\RAG\Yas\wordfiles"
output_folder = r"C:\Users\Admin\Desktop\RAG\Yas\pdffiles"
convert(input_folder, output_folder)

  0%|          | 0/66 [00:00<?, ?it/s]

## Implementing RAG for customer

In [1]:
#import the library
#%pip install -U langchain-google-genai
#%pip install -U langchain-google-genai faiss-cpu pypdf
from langchain_google_genai import ChatGoogleGenerativeAI

In [70]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.01,
    google_api_key="PUT_YOUR_GOOGLE_API_KEY_HERE"
)

In [71]:
# Generate answers to a question
question = "When did tanganyika get Independence?"
response = llm.invoke(question)
print(response.content)

Tanganyika gained independence on **December 9, 1961**.


In [4]:
# Generate answers to a question
question = "Who is the julius nyerere"
response = llm.invoke(question)
print(response.content)

**Julius Kambarage Nyerere** was a towering figure in African history, widely revered as the **first President of Tanzania** and affectionately known as **"Mwalimu"** (Swahili for 'Teacher'). He is considered the **"Father of the Nation" (Baba wa Taifa)** for his pivotal role in leading Tanganyika (which later merged with Zanzibar to form Tanzania) to independence and shaping its early development.

Here's a breakdown of who he was:

1.  **Early Life and Education:**
    *   Born in 1922 in Butiama, Tanganyika (then a British mandate).
    *   He received a mission education and later became the first Tanganyikan to study at a British university (Edinburgh University), where he earned a Master of Arts degree in history and economics.

2.  **Path to Independence:**
    *   Upon his return to Tanganyika, he became a teacher, but quickly became involved in politics.
    *   In 1954, he co-founded the **Tanganyika African National Union (TANU)**, a political party dedicated to achieving in

In [5]:
import os
import pickle
from langchain_google_genai import ChatGoogleGenerativeAI  #LLM (Google Gemini)
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # Embeddings (important for RAG)
from langchain_community.vectorstores import FAISS   # Vector DB
from langchain_community.document_loaders import PyPDFLoader    # Document loading
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Text splitting

C:\Users\Admin\AppData\Local\Temp\ipykernel_13828\2915587869.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS   # Vector DB


In [6]:
pdf_folder = r"C:\Users\Admin\Desktop\RAG\Yas\pdffiles"  # folder containing all PDFs
# Load ALL PDFs
documents = []
for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        file_path = os.path.join(pdf_folder, file)
        loader = PyPDFLoader(file_path)
        documents.extend(loader.load())
# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Total documents pages: {len(documents)}")
print(f"Total chunks created: {len(chunks)}")

Total documents pages: 549
Total chunks created: 2260


In [7]:
# ONLY FOR 1000CHUNK FOR A DAY
'''import time
import numpy as np

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2"
)

# Embed in small batches with delay
all_embeddings = []
batch_size = 90

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i+batch_size]
    batch_texts = [doc.page_content for doc in batch]
    batch_embeddings = embeddings.embed_documents(batch_texts)
    all_embeddings.extend(batch_embeddings)
    print(f"Processed {min(i+batch_size, len(chunks))}/{len(chunks)} chunks")
    if i + batch_size < len(chunks):  # no need to sleep after last batch
        time.sleep(65)  # slightly over 60s to be safe

print(f"Total embeddings created: {len(all_embeddings)}")

# Now build FAISS index from the pre-computed embeddings
texts = [doc.page_content for doc in chunks]
metadatas = [doc.metadata for doc in chunks]

db = FAISS.from_embeddings(
    text_embeddings=list(zip(texts, all_embeddings)),
    embedding=embeddings,
    metadatas=metadatas
)

print("FAISS index created successfully!")'''

'import time\nimport numpy as np\n\nembeddings = GoogleGenerativeAIEmbeddings(\n    model="models/gemini-embedding-2"\n)\n\n# Embed in small batches with delay\nall_embeddings = []\nbatch_size = 90\n\nfor i in range(0, len(chunks), batch_size):\n    batch = chunks[i:i+batch_size]\n    batch_texts = [doc.page_content for doc in batch]\n    batch_embeddings = embeddings.embed_documents(batch_texts)\n    all_embeddings.extend(batch_embeddings)\n    print(f"Processed {min(i+batch_size, len(chunks))}/{len(chunks)} chunks")\n    if i + batch_size < len(chunks):  # no need to sleep after last batch\n        time.sleep(65)  # slightly over 60s to be safe\n\nprint(f"Total embeddings created: {len(all_embeddings)}")\n\n# Now build FAISS index from the pre-computed embeddings\ntexts = [doc.page_content for doc in chunks]\nmetadatas = [doc.metadata for doc in chunks]\n\ndb = FAISS.from_embeddings(\n    text_embeddings=list(zip(texts, all_embeddings)),\n    embedding=embeddings,\n    metadatas=meta

In [8]:
'''#%pip install -U google-generativeai
import google.generativeai as genai
os.environ["GOOGLE_API_KEY"] = "PUT_YOUR_GOOGLE_API_KEY_HERE"
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)'''

'#%pip install -U google-generativeai\nimport google.generativeai as genai\nos.environ["GOOGLE_API_KEY"] = "PUT_YOUR_GOOGLE_API_KEY_HERE"\ngenai.configure(api_key=os.environ["GOOGLE_API_KEY"])\nfor m in genai.list_models():\n    if "embedContent" in m.supported_generation_methods:\n        print(m.name)'

In [9]:
#%pip install -U langchain-huggingface
#%pip install -U sentence-transformers
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Runs fully locally — no API, no quota, no waiting
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

db.save_local(r"C:\Users\Admin\Desktop\RAG\Yas\faiss")
print("FAISS index saved successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index saved successfully!


In [95]:
# Initialize Gemin model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.01,
    google_api_key="PUT_YOUR_GOOGLE_API_KEY_HERE"
)

In [103]:
import time
import requests
from bs4 import BeautifulSoup
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser


# ─── Constants ─────────────────────────────────────────────────────────────────

NO_ANSWER_MESSAGE = (
    "We could not locate the requested information in our available resources.\n\n"
    "For additional support, please 🙏 contact the YAS Customer Care team using the details below.\n\n"
    "📞 YAS Customers: Dial 100 directly from your YAS line.\n"
    "☎️ Customer Care: Dial 101 or +255 711 100 101 for support and inquiries.\n"
    "💬 WhatsApp Support: +255 714 100 100 for quick assistance.\n"
    "🌍 International Roaming Support: +255 714 100 100.\n\n"
     "Our Customer Care team is available to assist you with inquiries, complaints, technical support, and other YAS services"
)

REFUSAL_MESSAGE = (
    "Sorry, I can only answer questions about YAS Tanzania,🙏 "
    "such as bundles, airtime, SIM cards, roaming, fiber, offers, and customer support."
)
YAS_KEYWORDS = [
    "yas", "myyas", "mixx by yas", "yas tanzania",
    "simcard", "sim card", "bundle", "data bundle", "airtime",
    "recharge", "topup", "top up", "minutes", "sms", "call",
    "network", "coverage", "plan", "package", "offer",
    "subscription", "internet bundle", "internet package",
    "4g", "5g", "roaming", "balance", "prepaid", "postpaid",
    "billing", "invoice", "activation", "deactivation",
    "esim", "ussd", "fiber", "wakishua"
]

YAS_BRAND_WORDS = [
    "yas", "myyas", "mixx by yas", "yas tanzania"
]

YAS_ACTION_WORDS = [
    "buy", "subscribe", "activate", "deactivate", "check",
    "dial", "pay", "recharge", "top up", "use", "get",
    "how much", "price", "cost", "validity", "contact"
]

YAS_COMPANY_WORDS = [
    "your ceo", "yas ceo", "ceo of yas", "about yas",
    "yas company", "yas support", "yas customer care"
]


# ─── Domain Check ──────────────────────────────────────────────────────────────

def is_yas_related(question: str) -> bool:
    q = question.lower().strip()

    if any(kw in q for kw in YAS_BRAND_WORDS):
        return True

    if any(kw in q for kw in YAS_COMPANY_WORDS):
        return True

    has_yas_service = any(kw in q for kw in YAS_KEYWORDS)
    has_yas_action = any(kw in q for kw in YAS_ACTION_WORDS)

    return has_yas_service and has_yas_action


# ─── Smart Page Selector ───────────────────────────────────────────────────────

def get_relevant_pages(question: str) -> list:
    """Select only pages likely to contain the answer — no need to scrape all 66."""
    q = question.lower()
    selected = set()

    selected.add("https://www.yas.co.tz/")

    rules = {
        ("bundle", "data bundle", "internet", "gb", "mb"):
            ["https://www.yas.co.tz/consumer/mobile-plans/internet/",
             "https://www.yas.co.tz/consumer/mobile-plans/mix/",
             "https://www.yas.co.tz/consumer/mobile-plans/best-deals/"],

        ("voice", "sms", "call", "minutes", "airtime"):
            ["https://www.yas.co.tz/consumer/mobile-plans/voice-sms/",
             "https://www.yas.co.tz/consumer/mobile-plans/mix/"],

        ("international", "roaming"):
            ["https://www.yas.co.tz/consumer/mobile-plans/international/",
             "https://www.yas.co.tz/consumer/mobile-plans/roaming/"],

        ("fiber", "home", "broadband", "wifi"):
            ["https://www.yas.co.tz/fiber-home/",
             "https://www.yas.co.tz/consumer/home-plans/fiber-home/"],

        ("business", "sme", "corporate", "enterprise"):
            ["https://www.yas.co.tz/business/",
             "https://www.yas.co.tz/business/pro-sme/mobile/"],

        ("about yas", "yas ceo", "ceo of yas", "yas company", "yas director", "yas history"):
            ["https://www.yas.co.tz/about/",
             "https://www.yas.co.tz/about-yas-faqs/"],

        ("yas support", "yas customer care", "contact yas", "assistance"):
            ["https://www.yas.co.tz/assistance",
             "https://www.yas.co.tz/consumer-faqs",
             "https://www.yas.co.tz/about-yas-faqs/"],

        ("device", "phone", "smartphone", "zte", "tecno"):
            ["https://www.yas.co.tz/devices/"],

        ("store", "location", "shop", "branch"):
            ["https://www.yas.co.tz/store-locator/"],

        ("mixx", "lifestyle"):
            ["https://www.yas.co.tz/mixx-by-yas/",
             "https://www.yas.co.tz/consumer/lifestyle/"],

        ("wakishua", "offer", "deal", "promotion"):
            ["https://www.yas.co.tz/consumer/mobile-plans/wakishua/",
             "https://www.yas.co.tz/consumer/mobile-plans/best-deals/"],
    }

    for keywords, pages in rules.items():
        if any(kw in q for kw in keywords):
            selected.update(pages)

    return list(selected)


# ─── Fast Website Scraper ──────────────────────────────────────────────────────

def search_yas_website(question: str) -> str:
    """Scrape only relevant pages — fast and accurate."""
    pages = get_relevant_pages(question)
    print(f"🌐Scraping {len(pages)} relevant YAS pages...")

    all_text = ""
    headers = {"User-Agent": "Mozilla/5.0"}

    for url in pages:
        try:
            response = requests.get(url, headers=headers, timeout=5)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, "html.parser")
                for tag in soup(["script", "style", "nav", "footer"]):
                    tag.decompose()
                text = soup.get_text(separator=" ", strip=True)
                all_text += f"\n\n[Source: {url}]\n{text[:2000]}"
        except:
            continue

    return all_text.strip() if all_text else ""


# ─── Prompts ───────────────────────────────────────────────────────────────────

RAG_PROMPT = PromptTemplate.from_template("""You are an AI assistant for YAS telecom company in Tanzania.
Answer ONLY using the context below. Do not use outside knowledge.
Only answer questions related to YAS Tanzania.
If the answer is not in the context, say exactly "NOT_IN_DOCUMENT".

Context: {context}
Chat History: {chat_history}
Question: {question}

Answer:""")

WEBSITE_PROMPT = PromptTemplate.from_template("""You are an AI assistant for YAS telecom company in Tanzania.
Answer using ONLY the website content below. Do not use outside knowledge.
Only answer questions related to YAS Tanzania.
If the answer is not in the content, say exactly "NOT_ON_WEBSITE".

Website Content: {website_content}
Chat History: {chat_history}
Question: {question}

Answer:""")


# ─── Chains ────────────────────────────────────────────────────────────────────

retriever = db.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": RunnableLambda(lambda x: format_docs(retriever.invoke(x["question"]))),
        "question": RunnableLambda(lambda x: x["question"]),
        "chat_history": RunnableLambda(lambda x: x.get("chat_history", ""))
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

chat_history = []


# ─── Main Ask Function ─────────────────────────────────────────────────────────

def ask(question: str, similarity_threshold: float = 1.2) -> str:

    # Step 1: Domain check
    if not is_yas_related(question):
        print("🚫 Off-topic blocked.")
        return REFUSAL_MESSAGE
        

    # Step 2: Check FAISS score
    docs_with_scores = db.similarity_search_with_score(question, k=3)
    best_score = docs_with_scores[0][1] if docs_with_scores else 999
    print(f"FAISS score: {best_score:.4f} (threshold: {similarity_threshold})")

    # Step 3: Try FAISS first
    if best_score < similarity_threshold:
        print("📄 Searching in PDF documents...")
        answer = rag_chain.invoke({
            "question": question,
            "chat_history": chat_history
        }).strip()

        if "NOT_IN_DOCUMENT" not in answer:
            chat_history.append((question, answer))
            print(f"Source: 📄 PDF Document")
            print(f"Answer: {answer}\n")
            return answer

    # Step 4: Fallback to website
    print("⚠️ Information not found in PDF documents. Searching the YAS website...\n")
    website_content = search_yas_website(question)

    if website_content:
        website_chain = WEBSITE_PROMPT | llm | StrOutputParser()

        for attempt in range(3):
            try:
                answer = website_chain.invoke({
                    "website_content": website_content,
                    "question": question,
                    "chat_history": chat_history
                }).strip()

                if "NOT_ON_WEBSITE" not in answer:
                    chat_history.append((question, answer))
                    print("Source:🌐YAS Website")
                    print(f"Answer: {answer}\n")
                    return answer
                break

            except Exception as e:
                if "503" in str(e) or "UNAVAILABLE" in str(e):
                    wait = 10 * (attempt + 1)
                    print(f"⚠️ Gemini busy. Retrying in {wait}s... ({attempt+1}/3)")
                    time.sleep(wait)
                else:
                    raise e

    # Step 5: Nothing found
    chat_history.append((question, NO_ANSWER_MESSAGE))
    return NO_ANSWER_MESSAGE

In [104]:
print(ask("Who is the CEO of Yas & Mixx by Yas?"))

FAISS score: 0.8911 (threshold: 1.2)
📄 Searching in PDF documents...
⚠️ Information not found in PDF documents. Searching the YAS website...

🌐Scraping 5 relevant YAS pages...
Source:🌐YAS Website
Answer: Pierre Canton-Bacara is the CEO of Yas & Mixx by Yas.

Pierre Canton-Bacara is the CEO of Yas & Mixx by Yas.
